In [5]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv
from datetime import datetime
import requests
from pymongo import errors

# Carregar variáveis de ambiente
load_dotenv("/root/sic-conti-v2/sic-backend/.env")

# Inicializar o cliente MongoDB manualmente
class Mongo:
    def __init__(self, uri):
        self.client = MongoClient(uri)
        self.db = self.client.get_database()

# Instanciar a conexão com o banco
mongo = Mongo(os.getenv("MONGO_URI"))
TOKEN_RD = '65171203c468ec001837fdca'


In [9]:
#Lista as negociações e Atualiza a coleção negociações

def listar_negociacoes():
    headers = {"accept": "application/json"}
    offset = True
    page = 1
    limit = 200

    # Lista para armazenar os dados das negociações
    dados_negociacoes = []

    # Loop para obter todas as páginas da API
    while offset:
        url = f'https://crm.rdstation.com/api/v1/deals?token={TOKEN_RD}&page={page}&limit={limit}' #&win=true
        response = requests.get(url, headers=headers)

        # Verifica se a resposta foi bem-sucedida
        if response.status_code == 200:
            data = response.json()
            dados_negociacoes.extend(data['deals'])
            print(f"Página {page} - Mais negociações: {data['has_more']} - Total: {len(dados_negociacoes)}")

            # Atualiza o valor de offset e incrementa a página
            offset = data['has_more']
            page += 1
        else:
            print(f"Erro ao obter dados: {response.status_code}")
            break

    # Salvar os dados no arquivo JSON
    return dados_negociacoes
def atualizar_negociacoes(dados_negociacoes):
    # Selecionar a coleção
    colecao = mongo.db['negociacoes']
    
    # Atualizar os documentos na coleção
    for negociacao in dados_negociacoes:
        filtro = {"id": negociacao["id"]}
        atualizacao = {"$set": negociacao}
        try:
            colecao.update_one(filtro, atualizacao, upsert=True)
        except errors.PyMongoError as e:
            print(f"Erro ao atualizar documento {negociacao['id']}: {e}")

# Chamar a função para listar negociações e atualizar na coleção
negociacoes = listar_negociacoes()
atualizar_negociacoes(negociacoes)

Página 1 - Mais negociações: True - Total: 200
Página 2 - Mais negociações: True - Total: 400
Página 3 - Mais negociações: True - Total: 600
Página 4 - Mais negociações: True - Total: 800
Página 5 - Mais negociações: True - Total: 1000
Página 6 - Mais negociações: True - Total: 1200
Página 7 - Mais negociações: True - Total: 1400
Página 8 - Mais negociações: False - Total: 1408


In [10]:
def obter_novos_ganhos():
    """
    Retorna todas as negociações cujo campo 'win' é True e que não estão presentes na coleção 'warmup_projetos'.
    """
    pipeline = [
        {
            "$match": {
                "win": { "$eq": True }  # Filtrar negociações com win = true
            }
        },
        {
            "$lookup": {
                "from": "warmup_projetos",
                "localField": "id",
                "foreignField": "negocio_id",
                "as": "warmup_match"
            }
        },
        {
            "$match": {
                "warmup_match": []  # Manter apenas negociações sem correspondência
            }
        }
    ]
    
    # Executar a agregação e converter o resultado para uma lista
    resultado = list(mongo.db.negociacoes.aggregate(pipeline))
    return resultado
dados = obter_novos_ganhos()
len(dados)

783